In [1]:
!pip install -U deepeval
!pip install ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 812.5/812.5 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.37.0
    Uninstalling opentelemetry-proto-1.37.0:
      Successfully uninstalled opentelemetry-proto-1.37.0


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
%%bash
sudo apt-get update && sudo apt-get install zstd -y
curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 95 not upgraded.


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [16]:

# Create a persistent models directory in Google Drive
!mkdir -p /content/drive/MyDrive/RAG/ollama_models

# Remove any existing ephemeral models directory
!rm -rf ~/.ollama/models

# Symlink Ollama's model directory to Google Drive
!ln -s /content/drive/MyDrive/RAG/ollama_models ~/.ollama/models

print('Symlinked ~/.ollama/models -> /content/drive/MyDrive/RAG/ollama_models')

import subprocess, time

server = subprocess.Popen(['ollama', 'serve'])
print('Ollama server started as background process (PID:', server.pid, ')')
time.sleep(5)
print('Waited 5 seconds for the server to initialize.')



Symlinked ~/.ollama/models -> /content/drive/MyDrive/RAG/ollama_models
Ollama server started as background process (PID: 23458 )
Waited 5 seconds for the server to initialize.


In [17]:
# LLM
!ollama pull gpt-oss:20b

!ollama pull nomic-embed-text

In [18]:
import pandas as pd
import os

# Increase the timeout for DeepEval operations
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"

from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import OllamaModel

In [19]:
# DeepEval has a native Ollama integration (no OpenAI-compatible /v1 needed).
# Use the Ollama base URL WITHOUT /v1.
judge_llm = OllamaModel(
    model="gpt-oss:20b",
    base_url="http://localhost:11434",
    temperature=0,
    # Optional: pass Ollama options if you need a larger context window
    # (Ollama uses num_ctx rather than "max_tokens")
    generation_kwargs={"options": {"num_ctx": 128000}},
)

In [8]:
eval_dataset_df = pd.read_excel(r"/content/drive/MyDrive/RAG/results/eval_result_combined.xlsx", sheet_name="final")

eval_dataset_g1_df = eval_dataset_df[
    ["user_query", "ref_answer", "g1_response"]
].rename(
    columns={
        "user_query": "question",
        "g1_response": "answer",
        "ref_answer": "ground_truth",
    }
)

for col in ["question", "answer", "ground_truth"]:
    eval_dataset_g1_df[col] = eval_dataset_g1_df[col].astype(str)


In [9]:
from tqdm.auto import tqdm

correctness_metric = GEval(
    name="Answer Correctness",
    model=judge_llm,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    criteria=(
        "Score how correct the ACTUAL_OUTPUT is relative to the EXPECTED_OUTPUT.\n"
        "Consider factual correctness and completeness.\n"
        "If ACTUAL_OUTPUT contradicts EXPECTED_OUTPUT, score low.\n"
        "If ACTUAL_OUTPUT matches EXPECTED_OUTPUT in meaning (paraphrase is OK), score high.\n"
        "Ignore stylistic differences.\n"
    ),
    # Keep strict_mode=False to preserve 0–1 scoring behavior
    strict_mode=False,
    async_mode=False,   # safer in notebooks; set True if you want internal async execution
)

scores = []
reasons = []

for row in tqdm(eval_dataset_g1_df.itertuples(index=False), total=len(eval_dataset_g1_df)):
    tc = LLMTestCase(
        input=row.question,
        actual_output=row.answer,
        expected_output=row.ground_truth,
    )
    s = correctness_metric.measure(tc)
    scores.append(s)
    reasons.append(getattr(correctness_metric, "reason", None))

eval_dataset_g1_df["answer_correctness_score"] = scores
eval_dataset_g1_df["answer_correctness_reason"] = reasons  # optional but usually useful

display(eval_dataset_g1_df[["answer_correctness_score"]].head())


  0%|          | 0/197 [00:00<?, ?it/s]

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

,answer_correctness_score
0,1.0
1,1.0
2,0.0
3,0.2
4,0.4


In [10]:
eval_dataset_g1_df.to_excel(r"/content/drive/MyDrive/RAG/results/deepeval_g1_ac.xlsx", index=False)



In [11]:
eval_dataset_g1_df.head()

,question,ground_truth,answer,answer_correctness_score,answer_correctness_reason
0,Which month is set aside each year in the UAE ...,March,"Okay, let's break this down step by step to an...",1.0,The actual output correctly states that the mo...
1,"How many public, private, and charter schools ...","Abu Dhabi has 218 public schools, 203 private ...","Okay, let's break this down step by step to an...",1.0,The response accurately states the numbers for...
2,What mandatory early education requirement mus...,Austrian children must complete one compulsory...,"Okay, let's break this down step by step to an...",0.0,The actual answer states that six‑year‑old chi...
3,At what age does compulsory primary education ...,"Compulsory primary education begins at age 6, ...","Okay, let's analyze the context to answer your...",0.2,The response states that compulsory primary ed...
4,What purposes do Azerbaijan’s national and sch...,Both national centralized assessments and scho...,"Okay, let's break down how to answer this ques...",0.4,The response lists several purposes—individual...


## DeepSeek

In [20]:
eval_dataset_df = pd.read_excel(r"/content/drive/MyDrive/RAG/results/eval_result_combined.xlsx", sheet_name="final")

eval_dataset_d1_df = eval_dataset_df[
    ["user_query", "ref_answer", "d1_response"]
].rename(
    columns={
        "user_query": "question",
        "d1_response": "answer",
        "ref_answer": "ground_truth",
    }
)

for col in ["question", "answer", "ground_truth"]:
    eval_dataset_d1_df[col] = eval_dataset_d1_df[col].astype(str)


In [21]:
from tqdm.auto import tqdm

correctness_metric = GEval(
    name="Answer Correctness",
    model=judge_llm,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    criteria=(
        "Score how correct the ACTUAL_OUTPUT is relative to the EXPECTED_OUTPUT.\n"
        "Consider factual correctness and completeness.\n"
        "If ACTUAL_OUTPUT contradicts EXPECTED_OUTPUT, score low.\n"
        "If ACTUAL_OUTPUT matches EXPECTED_OUTPUT in meaning (paraphrase is OK), score high.\n"
        "Ignore stylistic differences.\n"
    ),
    # Keep strict_mode=False to preserve 0–1 scoring behavior
    strict_mode=False,
    async_mode=False,   # safer in notebooks; set True if you want internal async execution
)

scores = []
reasons = []

for row in tqdm(eval_dataset_d1_df.itertuples(index=False), total=len(eval_dataset_d1_df)):
    tc = LLMTestCase(
        input=row.question,
        actual_output=row.answer,
        expected_output=row.ground_truth,
    )
    s = correctness_metric.measure(tc)
    scores.append(s)
    reasons.append(getattr(correctness_metric, "reason", None))

eval_dataset_d1_df["answer_correctness_score"] = scores
eval_dataset_d1_df["answer_correctness_reason"] = reasons  # optional but usually useful

display(eval_dataset_d1_df[["answer_correctness_score"]].head())


  0%|          | 0/197 [00:00<?, ?it/s]

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

,answer_correctness_score
0,1.0
1,1.0
2,0.0
3,0.0
4,0.2


In [22]:
eval_dataset_d1_df.to_excel(r"/content/drive/MyDrive/RAG/results/deepeval_d1_ac_20260118.xlsx", index=False)